In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {DEVICE}")

BERT_MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
print(f"✅ Loaded tokenizer: {BERT_MODEL_NAME}")

In [ ]:
class DrugDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

y_raw = df["label"].values
X_raw = df["clean_text"].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_raw, y_raw, test_size=0.30, random_state=42, stratify=y_raw)
X_v,  X_te, y_v,  y_te  = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp)

BERT_BATCH = 16
train_ds = DrugDataset(list(X_tr), list(y_tr), bert_tokenizer)
val_ds   = DrugDataset(list(X_v),  list(y_v),  bert_tokenizer)
test_ds  = DrugDataset(list(X_te), list(y_te), bert_tokenizer)

train_loader = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BERT_BATCH)
test_loader  = DataLoader(test_ds,  batch_size=BERT_BATCH)

print(f"✅ Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")


In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=NUM_CLASSES
).to(DEVICE)
print(f"✅ Loaded BERT classifier: {BERT_MODEL_NAME}")

In [ ]:
EPOCHS = 3

optimizer = AdamW(bert_model.parameters(), lr=2e-5)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

def train_bert(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(data_loader, desc="Training BERT"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
    return total_loss / len(data_loader)

def evaluate_bert(model, data_loader, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating BERT"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            _, preds = torch.max(outputs.logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = correct_predictions.double() / len(data_loader.dataset)
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)

print("⏳ Training BERT model...")
for epoch in range(EPOCHS):
    train_loss = train_bert(bert_model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_accuracy, _, _ = evaluate_bert(bert_model, val_loader, DEVICE)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")

print("✅ BERT Training complete!")

In [ ]:
bert_model.eval()
bert_loss, bert_acc, bert_pred, bert_true = evaluate_bert(bert_model, test_loader, DEVICE)

print(f"\n{'='*50}")
print(f"📊 BERT Results")
print(f"{'='*50}")
print(f"🎯 Test Accuracy : {bert_acc*100:.2f}%")
print(f"📉 Test Loss     : {bert_loss:.4f}")

print("✅ bert_acc, bert_pred, and bert_true defined!")